# TinyML Lab 4 - Florence-2 results collector

Run this notebook **on the Raspberry Pi 5** after the Florence-2 environment is installed. It is organized around the lab's data-collection questions: captioning/detection, Portuguese OCR and grounding, and segmentation. Each experiment records timing, temperature when `vcgencmd` is available, memory snapshots, and the raw model result.

The original handout contains a few nested-quote f-string errors. This notebook corrects them and adds automatic CSV/JSON result saving.

## Before you run this

1. On the Pi, create and activate the `florence` virtual environment, then install `torch`, `torchvision`, `transformers==4.45.2`, `timm`, `einops`, `jupyter`, `Pillow`, `matplotlib`, and `opencv-python` as in the handout.
2. Create an `images/` folder beside this notebook and copy in `dogs-cats.jpg`, `table.jpg`, and `flyer.png` from the lab material.
3. Start Jupyter from the folder holding this notebook, open it with the `florence` Python environment, and run cells from top to bottom.
4. Keep `htop` open in another terminal during the complex segmentation test; the notebook captures before/after snapshots but cannot replace visual CPU monitoring.

In [ ]:
from pathlib import Path
from datetime import datetime
import copy, json, platform, random, shutil, subprocess, threading, time

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

IMAGE_DIR = Path('images')
RESULTS_DIR = Path('lab4_results')
RESULTS_DIR.mkdir(exist_ok=True)
MODEL_ID = 'microsoft/Florence-2-base'

IMAGE_PATHS = {
    'dogs_cats': IMAGE_DIR / 'dogs-cats.jpg',
    'table': IMAGE_DIR / 'table.jpg',
    'flyer': IMAGE_DIR / 'flyer.png',
}
missing = [str(path) for path in IMAGE_PATHS.values() if not path.exists()]
print('Notebook folder:', Path.cwd())
print('Missing images:', missing or 'none')

In [ ]:
# Run only if the packages are missing from the Florence virtual environment.
%pip install 'transformers==4.45.2' timm einops Pillow matplotlib opencv-python pandas requests

## 1. Load Florence-2

The first run downloads approximately 1-2 GB of model files and can take a while. Subsequent runs use the cached model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(f'PyTorch: {torch.__version__}')
print(f'Using device: {device}')

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch_dtype, trust_remote_code=True
).to(device)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model.eval()
print('Model loaded successfully.')

## 2. Helpers for inference, measurement, and visualisation

In [ ]:
def cpu_temperature_c():
    """Return Raspberry Pi CPU temperature, or None outside Raspberry Pi OS."""
    executable = shutil.which('vcgencmd')
    if not executable:
        return None
    try:
        text = subprocess.check_output([executable, 'measure_temp'], text=True).strip()
        return float(text.split('=')[1].split("'")[0])
    except Exception:
        return None

def memory_snapshot_mb():
    """Read total, used, and available RAM from `free -m`."""
    try:
        lines = subprocess.check_output(['free', '-m'], text=True).splitlines()
        values = lines[1].split()
        return {'total_mb': int(values[1]), 'used_mb': int(values[2]), 'available_mb': int(values[6])}
    except Exception:
        return {'total_mb': None, 'used_mb': None, 'available_mb': None}

class TemperatureSampler:
    def __init__(self, interval_s=1.0):
        self.interval_s = interval_s
        self.samples = []
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._sample, daemon=True)
    def _sample(self):
        while not self._stop.is_set():
            value = cpu_temperature_c()
            if value is not None:
                self.samples.append(value)
            self._stop.wait(self.interval_s)
    def start(self): self._thread.start()
    def stop(self):
        self._stop.set()
        self._thread.join(timeout=2)

def to_jsonable(value):
    if isinstance(value, np.ndarray): return value.tolist()
    if isinstance(value, np.generic): return value.item()
    if isinstance(value, dict): return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [to_jsonable(v) for v in value]
    return value

def run_florence_task(task_prompt, image, text_input='', max_new_tokens=1024):
    """Run one Florence-2 task and return parsed output plus lab measurements."""
    prompt = task_prompt + text_input
    temp_before = cpu_temperature_c()
    mem_before = memory_snapshot_mb()
    sampler = TemperatureSampler(); sampler.start()
    started = time.perf_counter()
    try:
        inputs = processor(text=prompt, images=image, return_tensors='pt').to(device, torch_dtype)
        with torch.inference_mode():
            generated_ids = model.generate(
                input_ids=inputs['input_ids'], pixel_values=inputs['pixel_values'],
                max_new_tokens=max_new_tokens, do_sample=False, num_beams=3,
            )
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
        parsed = processor.post_process_generation(
            generated_text, task=task_prompt, image_size=(image.width, image.height)
        )
    finally:
        elapsed_s = time.perf_counter() - started
        sampler.stop()
    temp_after = cpu_temperature_c()
    mem_after = memory_snapshot_mb()
    measurement = {
        'timestamp': datetime.now().isoformat(timespec='seconds'),
        'task': task_prompt, 'text_input': text_input, 'latency_s': round(elapsed_s, 3),
        'cpu_temp_before_c': temp_before, 'cpu_temp_peak_c': max(sampler.samples, default=temp_before),
        'cpu_temp_after_c': temp_after,
        'memory_before': mem_before, 'memory_after': mem_after,
        'result': to_jsonable(parsed),
    }
    print(f'{task_prompt} completed in {elapsed_s:.1f}s')
    return parsed, measurement

def plot_bbox(image, data, figsize=(12, 8)):
    fig, ax = plt.subplots(figsize=figsize); ax.imshow(image)
    for bbox, label in zip(data.get('bboxes', []), data.get('labels', [])):
        x1, y1, x2, y2 = bbox
        ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none'))
        ax.text(x1, y1, str(label), color='white', fontsize=9, bbox={'facecolor': 'red', 'alpha': .7})
    ax.axis('off'); plt.show()

COLORS = ['blue', 'orange', 'green', 'purple', 'brown', 'cyan', 'red', 'lime']
def draw_polygons(image, prediction, fill_mask=True):
    output = image.copy(); draw = ImageDraw.Draw(output)
    for polygons, label in zip(prediction.get('polygons', []), prediction.get('labels', [])):
        color = random.choice(COLORS)
        for polygon in polygons:
            points = np.array(polygon).reshape(-1).tolist()
            if len(points) >= 6:
                draw.polygon(points, outline=color, fill=color if fill_mask else None)
                draw.text((points[0] + 6, points[1] + 2), str(label), fill='white')
    return output

In [ ]:
dogs_cats = Image.open(IMAGE_PATHS['dogs_cats']).convert('RGB')
table = Image.open(IMAGE_PATHS['table']).convert('RGB')
flyer = Image.open(IMAGE_PATHS['flyer']).convert('RGB')

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, image) in zip(axes, [('dogs-cats', dogs_cats), ('table', table), ('flyer', flyer)]):
    ax.imshow(image); ax.set_title(name); ax.axis('off')
plt.show()

## 3. Question 2 - captioning and object detection on `dogs-cats.jpg`

Run all four tasks. The output table is saved as `lab4_results/q2_caption_detection.csv`; the complete raw outputs are saved in the JSON file.

In [ ]:
q2_tasks = ['<CAPTION>', '<DETAILED_CAPTION>', '<MORE_DETAILED_CAPTION>', '<OD>']
q2_rows = []
for task in q2_tasks:
    parsed, record = run_florence_task(task, dogs_cats)
    output = parsed[task]
    if task == '<OD>':
        labels = output.get('labels', [])
        record['summary'] = f"{len(labels)} detections: {labels}"
        record['detection_count'] = len(labels)
        plot_bbox(dogs_cats, output)
    else:
        record['summary'] = str(output)
        record['detection_count'] = None
        print(task, ':', output)
    q2_rows.append(record)

q2_df = pd.DataFrame([{
    'task': r['task'], 'latency_s': r['latency_s'],
    'temp_before_c': r['cpu_temp_before_c'], 'temp_peak_c': r['cpu_temp_peak_c'], 'temp_after_c': r['cpu_temp_after_c'],
    'memory_used_before_mb': r['memory_before']['used_mb'], 'memory_used_after_mb': r['memory_after']['used_mb'],
    'detection_count': r['detection_count'], 'output': r['summary']
} for r in q2_rows])
q2_df.to_csv(RESULTS_DIR / 'q2_caption_detection.csv', index=False)
(RESULTS_DIR / 'q2_caption_detection_raw.json').write_text(json.dumps(q2_rows, indent=2, ensure_ascii=False))
display(q2_df)
print('Fastest:', q2_df.loc[q2_df.latency_s.idxmin(), 'task'])
print('Slowest:', q2_df.loc[q2_df.latency_s.idxmax(), 'task'])

Fill in the real number of cats and dogs after inspecting the image, then compare it with the `<OD>` detection count.

In [ ]:
ACTUAL_CATS = None  # replace with your count
ACTUAL_DOGS = None  # replace with your count
od_labels = q2_rows[-1]['result'].get('<OD>', {}).get('labels', [])
print('Florence-2 labels:', od_labels)
print('Actual animals:', ACTUAL_CATS, 'cats and', ACTUAL_DOGS, 'dogs')

## 4. Question 3 - Portuguese OCR and visual grounding on `flyer.png`

In [ ]:
q3_rows = []
for task in ['<OCR>', '<OCR_WITH_REGION>']:
    parsed, record = run_florence_task(task, flyer)
    output = parsed[task]
    record['summary'] = str(output)
    record['region_count'] = len(output.get('labels', [])) if isinstance(output, dict) else None
    q3_rows.append(record)
    print(task, ':', output)
    if task == '<OCR_WITH_REGION>' and isinstance(output, dict):
        overlay = flyer.copy(); draw = ImageDraw.Draw(overlay)
        for box, label in zip(output.get('quad_boxes', []), output.get('labels', [])):
            points = np.array(box).reshape(-1).tolist(); draw.polygon(points, outline='red', width=3)
            draw.text((points[0], points[1]), str(label), fill='red')
        plt.figure(figsize=(10, 10)); plt.imshow(overlay); plt.axis('off'); plt.show()

for phrase in ['Inteligência Artificial', 'Raspberry Pi', 'Edge Computing']:
    parsed, record = run_florence_task('<CAPTION_TO_PHRASE_GROUNDING>', flyer, text_input=phrase)
    output = parsed['<CAPTION_TO_PHRASE_GROUNDING>']
    boxes = output.get('bboxes', []) if isinstance(output, dict) else []
    record['phrase'] = phrase
    record['found'] = bool(boxes)
    record['confidence_or_output'] = str(output)
    record['summary'] = str(output)
    record['region_count'] = len(boxes)
    q3_rows.append(record)
    print(f'{phrase}: found={bool(boxes)}; output={output}')
    if boxes: plot_bbox(flyer, output)

q3_df = pd.DataFrame([{
    'task': r['task'], 'phrase': r.get('phrase', ''), 'found': r.get('found', ''),
    'regions': r.get('region_count'), 'latency_s': r['latency_s'],
    'temp_peak_c': r['cpu_temp_peak_c'], 'output': r['summary']
} for r in q3_rows])
q3_df.to_csv(RESULTS_DIR / 'q3_portuguese_ocr_grounding.csv', index=False)
(RESULTS_DIR / 'q3_portuguese_ocr_grounding_raw.json').write_text(json.dumps(q3_rows, indent=2, ensure_ascii=False))
display(q3_df)

## 5. Question 4 - segmentation versus detection on `table.jpg`

Keep `htop` open while this cell runs. Segmentation can take several minutes per object on Raspberry Pi 5.

In [ ]:
objects = ['a wine bottle', 'a wine glass', 'a half orange']
q4_rows = []
for obj in objects:
    detection, det_record = run_florence_task('<CAPTION_TO_PHRASE_GROUNDING>', table, text_input=obj)
    det_output = detection['<CAPTION_TO_PHRASE_GROUNDING>']
    det_record.update({'object': obj, 'kind': 'grounding/detection', 'accurate_manual': None})
    q4_rows.append(det_record)
    if det_output.get('bboxes'): plot_bbox(table, det_output)

    segmentation, seg_record = run_florence_task('<REFERRING_EXPRESSION_SEGMENTATION>', table, text_input=obj)
    seg_output = segmentation['<REFERRING_EXPRESSION_SEGMENTATION>']
    seg_record.update({'object': obj, 'kind': 'segmentation', 'accurate_manual': None})
    q4_rows.append(seg_record)
    plt.figure(figsize=(8, 8)); plt.imshow(draw_polygons(table, seg_output)); plt.title(obj); plt.axis('off'); plt.show()

q4_df = pd.DataFrame([{
    'object': r['object'], 'task': r['task'], 'kind': r['kind'], 'latency_s': r['latency_s'],
    'temp_before_c': r['cpu_temp_before_c'], 'temp_peak_c': r['cpu_temp_peak_c'], 'temp_after_c': r['cpu_temp_after_c'],
    'memory_used_before_mb': r['memory_before']['used_mb'], 'memory_used_after_mb': r['memory_after']['used_mb'],
    'accurate_manual': r['accurate_manual']
} for r in q4_rows])
seg = q4_df[q4_df.kind == 'segmentation'].set_index('object')['latency_s']
det = q4_df[q4_df.kind == 'grounding/detection'].set_index('object')['latency_s']
q4_ratio = (seg / det).rename('segmentation_to_detection_ratio').reset_index()
q4_df.to_csv(RESULTS_DIR / 'q4_segmentation.csv', index=False)
q4_ratio.to_csv(RESULTS_DIR / 'q4_segmentation_ratio.csv', index=False)
(RESULTS_DIR / 'q4_segmentation_raw.json').write_text(json.dumps(q4_rows, indent=2, ensure_ascii=False))
display(q4_df)
display(q4_ratio)
print('Inspect each polygon visualisation and change accurate_manual to True/False before your report.')

## 6. Optional extra Florence-2 tasks

These are not required for Questions 2-4, but they are useful for screenshots and for exploring the lab material.

In [ ]:
# Example: dense region captioning
dense, dense_record = run_florence_task('<DENSE_REGION_CAPTION>', table)
plot_bbox(table, dense['<DENSE_REGION_CAPTION>'])
print(dense_record['result'])

# Example: open-vocabulary detection. Change the phrase as needed.
query = 'a wine bottle'
open_vocab, open_vocab_record = run_florence_task('<OPEN_VOCABULARY_DETECTION>', table, text_input=query)
open_vocab_data = open_vocab['<OPEN_VOCABULARY_DETECTION>']
od_style = {'bboxes': open_vocab_data.get('bboxes', []), 'labels': open_vocab_data.get('bboxes_labels', [])}
plot_bbox(table, od_style)
print(open_vocab_record['result'])

## 7. Final results bundle

Run this after the task cells. Upload the CSV files and selected screenshots only if your course requires them; no data is sent anywhere by this notebook.

In [ ]:
metadata = {
    'created_at': datetime.now().isoformat(), 'model': MODEL_ID,
    'platform': platform.platform(), 'python': platform.python_version(),
    'device': device, 'torch_version': torch.__version__,
    'images': {key: str(value) for key, value in IMAGE_PATHS.items()},
    'files': sorted(path.name for path in RESULTS_DIR.iterdir()),
}
(RESULTS_DIR / 'experiment_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved files:')
for path in sorted(RESULTS_DIR.iterdir()): print('-', path.name)